# Imports

In [ ]:
# %load_ext autoreload
# %autoreload 2
# %env ANYWIDGET_HMR=1
# from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.animation import FuncAnimation
from umap import UMAP

from dimbridge import Dimbridge

# %matplotlib inline
plt.style.use("ggplot")
plt.style.use("seaborn-v0_8-colorblind")

# Example 1: Synthetic Data, 4D parametrization of a Klein bottle 
https://en.wikipedia.org/wiki/Klein_bottle#4-D_non-intersecting

## Generate a Klein bottle dataset

In [ ]:
n = int(1e4)

## data
R = 2
P = 3
eps = 0.5
u = np.random.rand(n) * np.pi * 2
v = np.random.rand(n) * np.pi * 2

x = R * (np.cos(u / 2) * np.cos(v) - np.sin(u / 2) * np.sin(2 * v))
y = R * (np.sin(u / 2) * np.cos(v) + np.cos(u / 2) * np.sin(2 * v))
z = P * np.cos(u) * (1 + eps * np.sin(v))
w = P * np.sin(u) * (1 + eps * np.sin(v))

## construct pandas dataframe and compute UMAP
df = pd.DataFrame(dict(x1=x, x2=y, x3=z, x4=w))
# xy = UMAP(n_neighbors=50, min_dist=0.3).fit_transform(df.to_numpy())

# Here we will use the 2D parametrization of the bottle as our "dimensionality reduction" plot
xy = np.c_[u, v]

## validate UMAP
plt.figure(figsize=[12, 6], dpi=80)
plt.subplot(121)
plt.scatter(xy[:, 0], xy[:, 1], s=1)
plt.axis("equal")
plt.xlabel("u")
plt.ylabel("v")
plt.title("2D Parameter space")

## validate data
plt.subplot(122)
plt.scatter(df["x1"], df["x2"], s=1)
plt.axis("equal")
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("First 2 coordinates of the 4D embedding space")
plt.show()

print("Data table")
df

## Using DimBridge on Klein bottle dataset

In [ ]:
# for dev testing:
# from importlib import reload
# import dimbridge
# reload(dimbridge)

from dimbridge import Dimbridge

dimbridge = Dimbridge(
    data=df,  ## data table as a pandas DataFrame
    x=xy[:, 0],  ## x coordinate of DR plot
    y=xy[:, 1],  ## y coordinate of DR plot
    s=4,  # projection plot mark size
    splom_s=1,  # scatterplot matrix (SPLOM) mark size
    # "data extent" - displays min and max of selection in the predicate view,
    # "predicate regression" - uses ML method to balance false positives and false negatives
    predicate_mode="predicate regression",
    # Mode of the brush interaction on projection view, either 'single', "contrastive", or "curve"
    brush_mode="contrastive",
)
dimbridge

## Showing brush selected points

In [ ]:
## Showing the selected points in the first box
print("dimbridge.selected shape", np.array(dimbridge.selected).shape)
selected = np.array(dimbridge.selected[0])
df[selected]

## TODOs

In [ ]:
# similar to PyTorch's model.state_dict(), 
# returns a dict of states that is ready to torch.save(..)
dimbridge.state_dict() 

In [ ]:
dimbridge.data

In [ ]:
# state: <python_dict> or json, pth file name
# dimbridge.load_state_dict(state)